# 07 End-to-End Maintenance Pipeline & Export (Scale-Weighted Multi-Label)

This notebook builds a single Scikit-Learn `Pipeline` using `FunctionTransformer` and `PerTargetScaleWeightedClassifier` to predict machine failure modes (`twf`, `hdf`, `pwf`, `osf`, `rnf`) directly from raw sensor data, and exports the high-performance pipeline to `.pkl`.

## 1. Import Libraries & Load Raw Dataset

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.metrics import recall_score, precision_score, f1_score

sys.path.append(os.path.abspath(".."))
from src.data_loader import load_ai4i_data
from src.pipeline_utils import preprocess_cleaning, feature_engineering, PerTargetScaleWeightedClassifier, interpret_prediction

df = load_ai4i_data()
failure_cols = ['twf', 'hdf', 'pwf', 'osf', 'rnf']
X = df.drop(columns=['machine_failure', 'twf', 'hdf', 'pwf', 'osf', 'rnf'])
y = df[failure_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

Training set shape: (8000, 8), Testing set shape: (2000, 8)


## 2. Define Transformer Functions

In [2]:
# Preprocessing, Feature Engineering, and PerTargetScaleWeightedClassifier imported from src.pipeline_utils
# preprocess_cleaning(df_in)
# feature_engineering(df_in)
# PerTargetScaleWeightedClassifier()

## 3. Build & Train Scale-Weighted Multi-Label Pipeline

In [3]:
model_dir = '../models'
if not os.path.exists(model_dir):
    model_dir = 'models'
os.makedirs(model_dir, exist_ok=True)

pipeline = Pipeline(steps=[
    ('cleaning', FunctionTransformer(preprocess_cleaning)),
    ('feature_engineering', FunctionTransformer(feature_engineering)),
    ('scaler', StandardScaler()),
    ('classifier', PerTargetScaleWeightedClassifier())
])

pipeline.fit(X_train, y_train)
print("Scale-Weighted Multi-Label Pipeline training completed successfully.")

Scale-Weighted Multi-Label Pipeline training completed successfully.


## 4. Pipeline Evaluation

In [4]:
y_pred_pipe = pipeline.predict(X_test)
y_pred_df = pd.DataFrame(y_pred_pipe, columns=failure_cols)

for col in failure_cols:
    rec = recall_score(y_test[col], y_pred_df[col], zero_division=0)
    prec = precision_score(y_test[col], y_pred_df[col], zero_division=0)
    f1 = f1_score(y_test[col], y_pred_df[col], zero_division=0)
    print(f"Failure Mode [{col.upper()}]: Recall={rec:.4f}, Precision={prec:.4f}, F1={f1:.4f}")

Failure Mode [TWF]: Recall=0.2727, Precision=0.0811, F1=0.1250
Failure Mode [HDF]: Recall=1.0000, Precision=1.0000, F1=1.0000
Failure Mode [PWF]: Recall=0.8500, Precision=0.8947, F1=0.8718
Failure Mode [OSF]: Recall=1.0000, Precision=0.9000, F1=0.9474
Failure Mode [RNF]: Recall=0.0000, Precision=0.0000, F1=0.0000


## 5. Export Multi-Label Pipeline to PKL File

In [5]:
export_path = os.path.join(model_dir, 'smart_maintenance_pipeline.pkl')
joblib.dump(pipeline, export_path)
print(f"Multi-Label Pipeline exported successfully to: {export_path}")

Multi-Label Pipeline exported successfully to: c:\Users\VICTUS\OneDrive\Documents\Edwin's_Project\CompasFeast\smart_manufacturing_maintenance\models\smart_maintenance_pipeline.pkl


## 6. Raw Data Inference Verification (Failure Modes Mapping)

In [6]:
sample_raw_data = pd.DataFrame([{
    'udi': 1,
    'product_id': 'M14860',
    'type': 'M',
    'air_temperature_k': 298.1,
    'process_temperature_k': 308.6,
    'rotational_speed_rpm': 1551,
    'torque_nm': 42.8,
    'tool_wear_min': 0
}])

loaded_pipeline = joblib.load(export_path)
pred_raw = loaded_pipeline.predict(sample_raw_data)
status = interpret_prediction(pred_raw)

print(f"Predicted Raw Binary Output: {pred_raw[0]}")
print(f"Interpreted Machine Condition: {status}")

Predicted Raw Binary Output: [0 0 0 0 0]
Interpreted Machine Condition: Normal (No Failure Detected)
